# ARMD Stock 实验集成 Notebook

将项目中的原始代码按 main.py 执行逻辑集成，分为：
1. **import 依赖**
2. **全局变量 & 配置**
3. **辅助函数**——beta schedule、model_utils、io_utils、build_dataloader
4. **核心逻辑类**——Linear、ARMD、ReduceLROnPlateauWithWarmup、Trainer、CustomDataset
5. **实验**——加载配置、构建数据、训练、采样预测、评估 MSE/MAE

Author: guiying li
Date: 2025-06-24

---
## 1. import 依赖

In [1]:
import os
import math
import time
import random
import yaml
import json
import warnings
import importlib

import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F

from torch import nn
from torch.optim import Adam
from torch.nn.utils import clip_grad_norm_
from einops import reduce
from functools import partial
from pathlib import Path
from tqdm.auto import tqdm
from ema_pytorch import EMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")

---
## 2. 全局变量 & 配置

In [2]:
# ---------- 全局变量 ----------
pred_len = 96          # 预测长度（也是扩散总步数 T）
timesteps = 96         # 扩散步数（与预测长度对齐）

# ---------- 种子 ----------
def set_seed(seed):
    """设置随机种子以保证可复现性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(2023)

# ---------- 设备 ----------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


---
## 3. 辅助函数

In [3]:
# ======== 3.1 model_utils 中的辅助函数 ========

def exists(x):
    """检查变量是否存在"""
    return x is not None

def default(val, d):
    """如果 val 存在则返回 val，否则返回 d (或调用 d())"""
    if exists(val):
        return val
    return d() if callable(d) else d

def identity(t, *args, **kwargs):
    """恒等函数"""
    return t

def extract(a, t, x_shape):
    """从1D张量 a 中按时间索引 t 取值，并 reshape 以匹配 x_shape"""
    b, *_ = t.shape
    out = a.gather(-1, t)
    return out.reshape(b, *((1,) * (len(x_shape) - 1)))

def normalize_to_neg_one_to_one(x):
    """归一化到 [-1, 1]"""
    return x * 2 - 1

def unnormalize_to_zero_to_one(x):
    """从 [-1, 1] 反归一化"""
    return (x + 1) * 0.5

In [ ]:
# ======== 3.2 io_utils 中的辅助函数 ========

def load_yaml_config(path):
    """加载 YAML 配置文件"""
    with open(path, encoding="utf-8") as f:
        config = yaml.full_load(f)
    return config

def instantiate_from_config(config):
    """
    通过配置中的 'target' 路径实例化类，传入 'params' 作为参数。
    优先从 globals() 查找已定义的类（notebook 集成模式），
    找不到时回退到 importlib 动态导入。
    """
    if config is None:
        return None
    if not "target" in config:
        raise KeyError("Expected key `target` to instantiate.")
    module, cls_name = config["target"].rsplit(".", 1)
    # 优先从全局命名空间查找（notebook 内已定义的类）
    if cls_name in globals():
        cls = globals()[cls_name]
    else:
        cls = getattr(importlib.import_module(module, package=None), cls_name)
    return cls(**config.get("params", dict()))

def get_model_parameters_info(model):
    """统计模型可训练/非可训练参数数量（格式化输出）"""
    parameters = {'overall': {'trainable': 0, 'non_trainable': 0, 'total': 0}}
    for child_name, child_module in model.named_children():
        parameters[child_name] = {'trainable': 0, 'non_trainable': 0}
        for pn, p in child_module.named_parameters():
            if p.requires_grad:
                parameters[child_name]['trainable'] += p.numel()
            else:
                parameters[child_name]['non_trainable'] += p.numel()
        parameters[child_name]['total'] = parameters[child_name]['trainable'] + parameters[child_name]['non_trainable']
        parameters['overall']['trainable'] += parameters[child_name]['trainable']
        parameters['overall']['non_trainable'] += parameters[child_name]['non_trainable']
        parameters['overall']['total'] += parameters[child_name]['total']
    
    def format_number(num):
        K = 2**10
        M = 2**20
        G = 2**30
        if num > G:
            return f"{round(float(num)/G, 2)}G"
        elif num > M:
            return f"{round(float(num)/M, 2)}M"
        elif num > K:
            return f"{round(float(num)/K, 2)}K"
        else:
            return str(num)
    
    def format_dict(d):
        for k, v in d.items():
            if isinstance(v, dict):
                format_dict(v)
            else:
                d[k] = format_number(v)
    format_dict(parameters)
    return parameters

In [ ]:
# ======== 3.3 beta schedule ========

def linear_beta_schedule(timesteps):
    """
    Linear beta schedule for diffusion.
    """
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float64)

def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine beta schedule as proposed in
    https://openreview.net/forum?id=-NEXDKk8gZ
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float64)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)

In [ ]:
# ======== 3.4 CustomDataset & build_dataloader ========

def noise_mask(X, masking_ratio, lm=3, mode='separate', distribution='geometric', exclude_feats=None):
    """创建随机布尔掩码，0 表示该位置需要被遮罩预测（本实验不涉及）"""
    if exclude_feats is not None:
        exclude_feats = set(exclude_feats)
    if distribution == 'geometric':
        if mode == 'separate':
            mask = np.ones(X.shape, dtype=bool)
            for m in range(X.shape[1]):
                if exclude_feats is None or m not in exclude_feats:
                    mask[:, m] = geom_noise_mask_single(X.shape[0], lm, masking_ratio)
        else:
            mask = np.tile(np.expand_dims(geom_noise_mask_single(X.shape[0], lm, masking_ratio), 1), X.shape[1])
    else:
        if mode == 'separate':
            mask = np.random.choice(np.array([True, False]), size=X.shape, replace=True,
                                    p=(1 - masking_ratio, masking_ratio))
        else:
            mask = np.tile(np.random.choice(np.array([True, False]), size=(X.shape[0], 1), replace=True,
                                            p=(1 - masking_ratio, masking_ratio)), X.shape[1])
    return mask

def geom_noise_mask_single(L, lm, masking_ratio):
    """几何分布噪声掩码生成"""
    keep_mask = np.ones(L, dtype=bool)
    p_m = 1 / lm
    p_u = p_m * masking_ratio / (1 - masking_ratio)
    p = [p_m, p_u]
    state = int(np.random.rand() > masking_ratio)
    for i in range(L):
        keep_mask[i] = state
        if np.random.rand() < p[state]:
            state = 1 - state
    return keep_mask

class CustomDataset(Dataset):
    """
    自定义时间序列数据集，支持滑动窗口采样、z-score 标准化、
    train/test 划分（比例划分或时序三划分 70/10/20）。
    """
    def __init__(
        self, 
        name,
        data_root, 
        window=64, 
        proportion=0.8, 
        save2npy=True, 
        neg_one_to_one=True,
        seed=123,
        period='train',
        output_dir='./OUTPUT',
        predict_length=None,
        missing_ratio=None,
        style='separate', 
        distribution='geometric', 
        mean_mask_length=3,
        three_split=False,
        train_ratio=0.7,
        val_ratio=0.1,
        norm_on_train=False,
    ):
        super(CustomDataset, self).__init__()
        self.three_split = three_split
        self.train_ratio = float(train_ratio)
        self.val_ratio = float(val_ratio)
        self.norm_on_train = norm_on_train
        if self.three_split:
            assert period in ('train', 'val', 'test')
            assert self.train_ratio > 0 and self.val_ratio >= 0
            assert self.train_ratio + self.val_ratio < 1.0 - 1e-9
        else:
            assert period in ['train', 'test']
        self.name, self.pred_len, self.missing_ratio = name, predict_length, missing_ratio
        self.style, self.distribution, self.mean_mask_length = style, distribution, mean_mask_length
        self.rawdata, self.scaler = self.read_data(data_root, self.name)
        if self.norm_on_train:
            n_total = self.rawdata.shape[0]
            train_end = int(n_total * self.train_ratio)
            self.scaler = StandardScaler().fit(self.rawdata[:train_end])
        self.dir = os.path.join(output_dir, 'samples')
        os.makedirs(self.dir, exist_ok=True)

        self.window, self.period = window, period
        self.len, self.var_num = self.rawdata.shape[0], self.rawdata.shape[-1]
        self.sample_num_total = max(self.len - self.window + 1, 0)
        self.save2npy = save2npy
        self.auto_norm = False

        self.data = self.__normalize(self.rawdata)
        if self.three_split:
            train_w, val_w, test_w = self.__getsamples_three_split(self.data, seed)
            self.samples = {'train': train_w, 'val': val_w, 'test': test_w}[period]
        else:
            train, inference = self.__getsamples(self.data, proportion, seed)
            self.samples = train if period == 'train' else inference
        if period in ('test', 'val'):
            if missing_ratio is not None:
                self.masking = self.mask_data(seed)
            elif predict_length is not None:
                masks = np.ones(self.samples.shape)
                masks[:, -predict_length:, :] = 0
                self.masking = masks.astype(bool)
            else:
                raise NotImplementedError()
        self.sample_num = self.samples.shape[0]

    def __getsamples(self, data, proportion, seed):
        x = np.zeros((self.sample_num_total, self.window, self.var_num))
        for i in range(self.sample_num_total):
            x[i, :, :] = data[i:i+self.window, :]
        train_data, test_data = self.divide(x, proportion, seed)
        if self.save2npy:
            if 1 - proportion > 0:
                np.save(os.path.join(self.dir, f"{self.name}_ground_truth_{self.window}_test.npy"),
                        self.unnormalize(test_data))
            np.save(os.path.join(self.dir, f"{self.name}_ground_truth_{self.window}_train.npy"),
                    self.unnormalize(train_data))
            np.save(os.path.join(self.dir, f"{self.name}_norm_truth_{self.window}_train.npy"), train_data)
            if 1 - proportion > 0:
                np.save(os.path.join(self.dir, f"{self.name}_norm_truth_{self.window}_test.npy"), test_data)
        return train_data, test_data

    def __getsamples_three_split(self, data, seed):
        x = np.zeros((self.sample_num_total, self.window, self.var_num))
        for i in range(self.sample_num_total):
            x[i, :, :] = data[i:i+self.window, :]
        n = x.shape[0]
        t_end = int(np.ceil(n * self.train_ratio))
        v_end = int(np.ceil(n * (self.train_ratio + self.val_ratio)))
        train_data, val_data, test_data = x[:t_end], x[t_end:v_end], x[v_end:]
        if self.save2npy:
            for tag, d in [('train', train_data), ('val', val_data), ('test', test_data)]:
                np.save(os.path.join(self.dir, f"{self.name}_ground_truth_{self.window}_{tag}.npy"),
                        self.unnormalize(d))
                np.save(os.path.join(self.dir, f"{self.name}_norm_truth_{self.window}_{tag}.npy"), d)
        return train_data, val_data, test_data

    def normalize(self, sq):
        d = sq.reshape(-1, self.var_num)
        d = self.scaler.transform(d)
        if self.auto_norm:
            d = normalize_to_neg_one_to_one(d)
        return d.reshape(-1, self.window, self.var_num)

    def unnormalize(self, sq):
        d = self.__unnormalize(sq.reshape(-1, self.var_num))
        return d.reshape(-1, self.window, self.var_num)
    
    def __normalize(self, rawdata):
        data = self.scaler.transform(rawdata)
        if self.auto_norm:
            data = normalize_to_neg_one_to_one(data)
        return data

    def __unnormalize(self, data):
        if self.auto_norm:
            data = unnormalize_to_zero_to_one(data)
        return self.scaler.inverse_transform(data)
    
    @staticmethod
    def divide(data, ratio, seed=2023):
        size = data.shape[0]
        st0 = np.random.get_state()
        np.random.seed(seed)
        regular_train_num = int(np.ceil(size * ratio))
        id_rdm = np.arange(size)
        regular_data = data[id_rdm[:regular_train_num]]
        irregular_data = data[id_rdm[regular_train_num:]]
        np.random.set_state(st0)
        return regular_data, irregular_data

    @staticmethod
    def read_data(filepath, name=''):
        df = pd.read_csv(filepath, header=0)
        if name == 'etth':
            df.drop(df.columns[0], axis=1, inplace=True)
        data = df.values.astype(np.float64)
        scaler = StandardScaler()
        scaler.fit(data)
        return data, scaler
    
    def mask_data(self, seed=2023):
        masks = np.ones_like(self.samples)
        st0 = np.random.get_state()
        np.random.seed(seed)
        for idx in range(self.samples.shape[0]):
            mask = noise_mask(self.samples[idx], self.missing_ratio,
                              self.mean_mask_length, self.style, self.distribution)
            masks[idx] = mask
        if self.save2npy:
            np.save(os.path.join(self.dir, f"{self.name}_masking_{self.window}.npy"), masks)
        np.random.set_state(st0)
        return masks.astype(bool)

    def __getitem__(self, ind):
        x = self.samples[ind]
        if self.period in ('test', 'val'):
            return torch.from_numpy(x).float(), torch.from_numpy(self.masking[ind]).float()
        return torch.from_numpy(x).float()

    def __len__(self):
        return self.sample_num


def build_dataloader(config, args):
    """构建训练 DataLoader"""
    batch_size = config['dataloader']['batch_size']
    jud = config['dataloader']['shuffle']
    config['dataloader']['train_dataset']['params']['output_dir'] = args.save_dir
    dataset = instantiate_from_config(config['dataloader']['train_dataset'])
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=jud,
                                             num_workers=0, pin_memory=True, drop_last=jud)
    return {'dataloader': dataloader, 'dataset': dataset}

def build_dataloader_cond(config, args):
    """构建测试 DataLoader"""
    batch_size = config['dataloader']['sample_size']
    config['dataloader']['test_dataset']['params']['output_dir'] = args.save_dir
    if args.mode == 'infill':
        config['dataloader']['test_dataset']['params']['missing_ratio'] = args.missing_ratio
    elif args.mode == 'predict':
        config['dataloader']['test_dataset']['params']['predict_length'] = args.pred_len
    test_dataset = instantiate_from_config(config['dataloader']['test_dataset'])
    dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                                             num_workers=0, pin_memory=True, drop_last=False)
    return {'dataloader': dataloader, 'dataset': test_dataset}

---
## 4. 核心逻辑类

In [ ]:
# ======== 4.1 Linear 模块 ========
# ARMD 的核心 devolution 网络 R(·)：
#   1. 对每个特征维度独立做时间轴线性映射
#   2. 用 W(t) 加权混合输入 X^t 与距离预测 D
#   3. 训练时加小扰动增加多样性

class Linear(nn.Module):
    """
    ARMD 的核心去噪/演化网络。
    对每个特征维度独立做时间轴上的线性映射 (nn.Linear(T, T))，
    然后用可学习的 W(t) 加权混合输入与距离预测。
    """
    def __init__(
        self,
        n_feat,
        n_channel,
        w_grad=True,
        **kwargs
    ):
        super().__init__()
        self.linear = nn.Linear(n_channel, n_channel)
        self.betas = linear_beta_schedule(96)
        self.betas_dev = cosine_beta_schedule(96)
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_dev = 1. - self.betas_dev
        self.w = torch.nn.Parameter(torch.FloatTensor(self.alphas_cumprod.numpy()), requires_grad=w_grad)
        self.w_dev = torch.nn.Parameter(torch.FloatTensor(self.alphas_dev.numpy()), requires_grad=False)

    def forward(self, input_, t, training=True):
        noise = torch.randn_like(input_)
        if not training:
            noise = 0
        input_ += self.w_dev[t[0]] * noise
        # permute: (B, T, F) -> (B, F, T) -> Linear 对时间维 (T) 做映射 -> (B, T, F)
        x_tmp = self.linear(input_.permute(0, 2, 1)).permute(0, 2, 1)
        alpha = self.w[t[0]]
        # Eq.5: W(t)*X^t + (1-2W(t))*D  /  sqrt(1-W(t))
        output = (alpha * input_ + (1 - 2 * alpha) * x_tmp) / (1 - 1 * alpha) ** (1 / 2)
        output = output.to(torch.float32)
        return output

In [ ]:
# ======== 4.2 ARMD 扩散模型 ========
# 前向过程：滑动操作（Slide）—— 从未来序列逐步滑动到历史序列
# 反向过程：从历史序列迭代生成（预测）未来序列
# 损失：预测演化趋势 z^hat 与真实 z 之间的 L1/L2

class ARMD(nn.Module):
    """
    Auto-Regressive Moving Diffusion 模型。
    前向：用滑动操作作为"加噪"，从未来序列 X^0 滑动到历史序列 X^T
    反向：从历史序列开始，逐步"反演"到未来（即预测）
    训练：每个 batch 随机采 t，预测演化趋势 z^t
    """
    def __init__(
        self,
        seq_length,
        feature_size,
        n_layer_enc=3,
        n_layer_dec=6,
        d_model=None,
        timesteps=1000,
        sampling_timesteps=None,
        loss_type='l1',
        beta_schedule='cosine',
        n_heads=4,
        mlp_hidden_times=4,
        eta=0.,
        attn_pd=0.,
        resid_pd=0.,
        w_grad=True,
        use_revin=False,
        **kwargs
    ):
        super(ARMD, self).__init__()
        self.eta = eta
        self.seq_length = seq_length
        self.feature_size = feature_size
        self.use_revin = use_revin
        self.model = Linear(n_feat=feature_size, n_channel=seq_length, w_grad=w_grad, **kwargs)
        
        if beta_schedule == 'linear':
            betas = linear_beta_schedule(timesteps)
        elif beta_schedule == 'cosine':
            betas = cosine_beta_schedule(timesteps)
        else:
            raise ValueError(f'unknown beta schedule {beta_schedule}')
        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.)
        timesteps, = betas.shape
        self.num_timesteps = int(timesteps)
        self.loss_type = loss_type
        self.sampling_timesteps = default(sampling_timesteps, timesteps)
        assert self.sampling_timesteps <= timesteps
        self.fast_sampling = self.sampling_timesteps < timesteps

        register_buffer = lambda name, val: self.register_buffer(name, val.to(torch.float32))
        register_buffer('betas', betas)
        register_buffer('alphas_cumprod', alphas_cumprod)
        register_buffer('alphas_cumprod_prev', alphas_cumprod_prev)
        register_buffer('sqrt_alphas_cumprod', torch.sqrt(alphas_cumprod))
        register_buffer('sqrt_one_minus_alphas_cumprod', torch.sqrt(1. - alphas_cumprod))
        register_buffer('log_one_minus_alphas_cumprod', torch.log(1. - alphas_cumprod))
        register_buffer('sqrt_recip_alphas_cumprod', torch.sqrt(1. / alphas_cumprod))
        register_buffer('sqrt_recipm1_alphas_cumprod', torch.sqrt(1. / alphas_cumprod - 1))

        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        register_buffer('posterior_variance', posterior_variance)
        register_buffer('posterior_log_variance_clipped', torch.log(posterior_variance.clamp(min=1e-20)))
        register_buffer('posterior_mean_coef1', betas * torch.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod))
        register_buffer('posterior_mean_coef2', (1. - alphas_cumprod_prev) * torch.sqrt(alphas) / (1. - alphas_cumprod))
        register_buffer('loss_weight', torch.sqrt(alphas) * torch.sqrt(1. - alphas_cumprod) / betas / 100)

    def predict_noise_from_start(self, x_t, t, x0):
        return ((extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t - x0) /
                extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape))
    
    def predict_start_from_noise(self, x_t, t, noise):
        return (extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t -
                extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise)

    def q_posterior(self, x_start, x_t, t):
        posterior_mean = (extract(self.posterior_mean_coef1, t, x_t.shape) * x_start +
                          extract(self.posterior_mean_coef2, t, x_t.shape) * x_t)
        posterior_variance = extract(self.posterior_variance, t, x_t.shape)
        posterior_log_variance_clipped = extract(self.posterior_log_variance_clipped, t, x_t.shape)
        return posterior_mean, posterior_variance, posterior_log_variance_clipped
    
    def output(self, x, t, training=False):
        return self.model(x, t, training=training)

    def model_predictions(self, x, t, clip_x_start=False, training=False):
        if training:
            training = False
        x_start = self.output(x, t, training)
        x_start = torch.clamp(x_start, min=-2, max=2) if clip_x_start else x_start
        pred_noise = self.predict_noise_from_start(x, t, x_start)
        return pred_noise, x_start

    def p_mean_variance(self, x, t, clip_denoised=True):
        _, x_start = self.model_predictions(x, t)
        if clip_denoised:
            x_start.clamp_(-1., 1.)
        model_mean, posterior_variance, posterior_log_variance = \
            self.q_posterior(x_start=x_start, x_t=x, t=t)
        return model_mean, posterior_variance, posterior_log_variance, x_start

    def p_sample(self, x, t: int, clip_denoised=True):
        batched_times = torch.full((x.shape[0],), t, device=x.device, dtype=torch.long)
        model_mean, _, model_log_variance, x_start = \
            self.p_mean_variance(x=x, t=batched_times, clip_denoised=clip_denoised)
        noise = torch.randn_like(x) if t > 0 else 0.
        pred_img = model_mean + (0.5 * model_log_variance).exp() * noise
        return pred_img, x_start

    @torch.no_grad()
    def sample(self, x):
        img = x[:, :pred_len, :]
        for t in tqdm(reversed(range(0, self.num_timesteps)),
                      desc='sampling loop time step', total=self.num_timesteps):
            img, _ = self.p_sample(img, t)
        return img
    
    @torch.no_grad()
    def fast_sample(self, x, clip_denoised=True):
        """DDIM 风格快速采样。当 sampling_timesteps=1 时一步直达预测。"""
        shape = x.shape
        batch, device, total_timesteps, sampling_timesteps, eta = \
            shape[0], self.betas.device, self.num_timesteps, self.sampling_timesteps, self.eta
        times = torch.linspace(-1, total_timesteps - 1, steps=sampling_timesteps + 1)
        times = list(reversed(times.int().tolist()))
        time_pairs = list(zip(times[:-1], times[1:]))
        img = x[:, :pred_len, :]
        for time, time_next in tqdm(time_pairs, desc='sampling loop time step'):
            time_cond = torch.full((batch,), time, device=device, dtype=torch.long)
            pred_noise, x_start, *_ = self.model_predictions(img, time_cond, clip_x_start=clip_denoised)
            if time_next < 0:
                img = x_start
                continue
            alpha = self.alphas_cumprod[time]
            alpha_next = self.alphas_cumprod[time_next]
            c = (1 - alpha_next).sqrt()
            noise = 0
            img = x_start * alpha_next.sqrt() + c * pred_noise + noise
        return img

    def _revin_stats(self, x, eps=1e-5):
        ctx = x[:, :pred_len, :]
        mu = ctx.mean(dim=1, keepdim=True)
        sigma = torch.sqrt(ctx.var(dim=1, keepdim=True, unbiased=False) + eps)
        return mu, sigma

    def generate_mts(self, x):
        """推理入口。x: (B, 192, F) -> 返回: (B, 96, F) 预测"""
        sample_fn = self.fast_sample if self.fast_sampling else self.sample
        if self.use_revin:
            mu, sigma = self._revin_stats(x)
            pred = sample_fn((x - mu) / sigma)
            return pred * sigma + mu
        return sample_fn(x)

    @property
    def loss_fn(self):
        if self.loss_type == 'l1':
            return F.l1_loss
        elif self.loss_type == 'l2':
            return F.mse_loss
        else:
            raise ValueError(f'invalid loss type {self.loss_type}')

    def q_sample(self, x_start, t, noise=None):
        """前向"加噪"：滑动操作。x_start 形状 (B, 192, F)"""
        index = int(t[0]) + 1
        x_middle = x_start[:, pred_len - index:-index, :]
        return x_middle

    def _train_loss(self, x_start, t, target=None, noise=None, training=True):
        """训练损失：预测演化趋势 z^hat vs 真实 z"""
        noise = default(noise, lambda: torch.randn_like(x_start))
        if target is None:
            target = x_start[:, pred_len:, :]
        target = x_start[:, pred_len:, :]
        x = self.q_sample(x_start=x_start, t=t, noise=noise)
        model_out = self.output(x, t, training)
        alpha = self.sqrt_alphas_cumprod[t[0]]
        minus_alpha = self.sqrt_one_minus_alphas_cumprod[t[0]]
        target_noise = (x - target * alpha) / minus_alpha
        pred_noise = (x - model_out * alpha) / minus_alpha
        train_loss = self.loss_fn(pred_noise, target_noise, reduction='none')
        train_loss = reduce(train_loss, 'b ... -> b (...)', 'mean')
        train_loss = train_loss * extract(self.loss_weight, t, train_loss.shape)
        return train_loss.mean()

    def forward(self, x, **kwargs):
        """训练入口。x: (B, 192, F) — [历史96 | 未来96]"""
        b, c, n, device, feature_size, = *x.shape, x.device, self.feature_size
        assert n == feature_size, f'number of variable must be {feature_size}'
        if self.use_revin:
            mu, sigma = self._revin_stats(x)
            x = (x - mu) / sigma
        t = torch.randint(0, self.num_timesteps, (1,), device=device).repeat(b).long()
        return self._train_loss(x_start=x, t=t, **kwargs)

In [ ]:
# ======== 4.3 ReduceLROnPlateauWithWarmup ========

class ReduceLROnPlateauWithWarmup:
    """带 warmup 的 ReduceLROnPlateau 学习率调度器"""
    def __init__(self, optimizer, mode='min', factor=0.1, patience=10,
                 threshold=1e-4, threshold_mode='rel', cooldown=0,
                 min_lr=0, eps=1e-8, verbose=False, warmup_lr=None, warmup=0):
        if factor >= 1.0:
            raise ValueError('Factor should be < 1.0.')
        self.factor = factor
        self.optimizer = optimizer
        self.min_lrs = [min_lr] * len(optimizer.param_groups) if isinstance(min_lr, (int, float)) else list(min_lr)
        self.patience = patience
        self.verbose = verbose
        self.cooldown = cooldown
        self.cooldown_counter = 0
        self.mode = mode
        self.threshold = threshold
        self.threshold_mode = threshold_mode
        self.warmup_lr = warmup_lr
        self.warmup = warmup
        self.best = None
        self.num_bad_epochs = None
        self.mode_worse = None
        self.eps = eps
        self.last_epoch = 0
        self._init_is_better(mode=mode, threshold=threshold, threshold_mode=threshold_mode)
        self._reset()

    def _prepare_for_warmup(self):
        if self.warmup_lr is not None:
            self.warmup_lrs = [self.warmup_lr] * len(self.optimizer.param_groups) if isinstance(self.warmup_lr, (int, float)) else list(self.warmup_lr)
        else:
            self.warmup_lrs = None
        if self.warmup > self.last_epoch:
            curr_lrs = [group['lr'] for group in self.optimizer.param_groups]
            self.warmup_lr_steps = [max(0, (self.warmup_lrs[i] - curr_lrs[i]) / float(self.warmup)) for i in range(len(curr_lrs))]
        else:
            self.warmup_lr_steps = None

    def _reset(self):
        self.best = self.mode_worse
        self.cooldown_counter = 0
        self.num_bad_epochs = 0

    def step(self, metrics):
        current = float(metrics)
        epoch = self.last_epoch + 1
        self.last_epoch = epoch
        if epoch <= self.warmup:
            self._increase_lr(epoch)
        else:
            if self.is_better(current, self.best):
                self.best = current
                self.num_bad_epochs = 0
            else:
                self.num_bad_epochs += 1
            if self.in_cooldown:
                self.cooldown_counter -= 1
                self.num_bad_epochs = 0
            if self.num_bad_epochs > self.patience:
                self._reduce_lr(epoch)
                self.cooldown_counter = self.cooldown
                self.num_bad_epochs = 0

    def _reduce_lr(self, epoch):
        for i, param_group in enumerate(self.optimizer.param_groups):
            old_lr = float(param_group['lr'])
            new_lr = max(old_lr * self.factor, self.min_lrs[i])
            if old_lr - new_lr > self.eps:
                param_group['lr'] = new_lr

    def _increase_lr(self, epoch):
        for i, param_group in enumerate(self.optimizer.param_groups):
            old_lr = float(param_group['lr'])
            new_lr = max(old_lr + self.warmup_lr_steps[i], self.min_lrs[i])
            param_group['lr'] = new_lr

    @property
    def in_cooldown(self):
        return self.cooldown_counter > 0

    def is_better(self, a, best):
        if self.mode == 'min' and self.threshold_mode == 'rel':
            return a < best * (1. - self.threshold)
        elif self.mode == 'min' and self.threshold_mode == 'abs':
            return a < best - self.threshold
        elif self.mode == 'max' and self.threshold_mode == 'rel':
            return a > best * (self.threshold + 1.)
        else:
            return a > best + self.threshold

    def _init_is_better(self, mode, threshold, threshold_mode):
        if mode == 'min':
            self.mode_worse = float('inf')
        else:
            self.mode_worse = -float('inf')
        self.mode = mode
        self.threshold = threshold
        self.threshold_mode = threshold_mode
        self._prepare_for_warmup()

In [ ]:
# ======== 4.4 Trainer ========

def cycle(dl):
    """无限循环的数据迭代器"""
    while True:
        for data in dl:
            yield data

class Trainer:
    """
    ARMD 训练器。
    - Adam 优化器 + EMA 指数移动平均
    - ReduceLROnPlateauWithWarmup 调度器
    - 梯度裁剪 + 周期性保存 checkpoint
    """
    def __init__(self, config, args, model, dataloader):
        self.model = model
        self.device = self.model.betas.device
        self.train_num_steps = config['solver']['max_epochs']
        self.gradient_accumulate_every = config['solver']['gradient_accumulate_every']
        self.save_cycle = config['solver']['save_cycle']
        self.dl = cycle(dataloader['dataloader'])
        self.step = 0
        self.milestone = 0
        self.args = args
        self.results_folder = Path(config['solver']['results_folder'] + f'_{model.seq_length}')
        os.makedirs(self.results_folder, exist_ok=True)

        start_lr = config['solver'].get('base_lr', 1.0e-4)
        ema_decay = config['solver']['ema']['decay']
        ema_update_every = config['solver']['ema']['update_interval']
        self.opt = Adam(filter(lambda p: p.requires_grad, self.model.parameters()),
                        lr=start_lr, betas=[0.9, 0.96])
        self.ema = EMA(self.model, beta=ema_decay, update_every=ema_update_every).to(self.device)
        sc_cfg = config['solver']['scheduler']
        sc_cfg['params']['optimizer'] = self.opt
        self.sch = instantiate_from_config(sc_cfg)

    def save(self, milestone):
        data = {'step': self.step, 'model': self.model.state_dict(),
                'ema': self.ema.state_dict(), 'opt': self.opt.state_dict()}
        torch.save(data, str(self.results_folder / f'checkpoint-{milestone}.pt'))

    def load(self, milestone):
        data = torch.load(str(self.results_folder / f'checkpoint-{milestone}.pt'), map_location=self.device)
        self.model.load_state_dict(data['model'])
        self.step = data['step']
        self.opt.load_state_dict(data['opt'])
        self.ema.load_state_dict(data['ema'])
        self.milestone = milestone

    def train(self):
        device = self.device
        step = 0
        with tqdm(initial=step, total=self.train_num_steps) as pbar:
            while step < self.train_num_steps:
                total_loss = 0.
                for _ in range(self.gradient_accumulate_every):
                    data = next(self.dl).to(device)
                    loss = self.model(data, target=data)
                    loss = loss / self.gradient_accumulate_every
                    loss.backward()
                    total_loss += loss.item()
                pbar.set_description(f'loss: {total_loss:.6f}')
                clip_grad_norm_(self.model.parameters(), 1.0)
                self.opt.step()
                self.sch.step(total_loss)
                self.opt.zero_grad()
                self.step += 1
                step += 1
                self.ema.update()
                with torch.no_grad():
                    if self.step != 0 and self.step % self.save_cycle == 0:
                        self.milestone += 1
                        self.save(self.milestone)
                pbar.update(1)
        print('training complete')

    def sample_forecast(self, raw_dataloader, shape=None):
        """使用 EMA 模型预测，返回 (samples, reals)"""
        samples = np.empty([0, shape[0], shape[1]])
        reals = np.empty([0, shape[0], shape[1]])
        for batch in raw_dataloader:
            x = batch[0] if len(batch) == 2 else batch
            x = x.to(self.device)
            sample = self.ema.ema_model.generate_mts(x)
            samples = np.row_stack([samples, sample.detach().cpu().numpy()])
            reals = np.row_stack([reals, x[:, shape[0]:, :].detach().cpu().numpy()])
            torch.cuda.empty_cache()
        return samples, reals

---
## 5. 实验：Stock 预测

按照 main.py 的执行逻辑：
1. 加载配置
2. 实例化模型
3. 构建训练 dataloader
4. 训练（2000 epoch）
5. 构建测试 dataloader
6. 采样预测 10 次，计算 MSE/MAE

In [ ]:
# ---------- 5.1 加载配置 & 参数 ----------

# 自动定位项目根目录（从当前工作目录向上查找 Config/）
_cwd = os.getcwd()
_project_root = _cwd
for _ in range(5):
    if os.path.isdir(os.path.join(_project_root, "Config")):
        break
    _project_root = os.path.dirname(_project_root)
else:
    raise FileNotFoundError('Cannot find Config/ directory from any ancestor. '
                            'Please run this notebook from the ARMD project root.')

# 切换到项目根目录，确保相对路径（./Data/, ./Config/ 等）正确解析
os.chdir(_project_root)
config_path = os.path.join(_project_root, 'Config', 'stock.yaml')
seq_len = 96

class Args:
    def __init__(self, config_path, save_dir, gpu):
        self.config_path = config_path
        self.save_dir = save_dir
        self.gpu = gpu
        os.makedirs(self.save_dir, exist_ok=True)

args = Args(config_path=str(config_path),
            save_dir=os.path.join(_project_root, 'forecasting_exp'),
            gpu=0)

configs = load_yaml_config(str(config_path))

print("Config loaded:")
print(f"  seq_length={configs['model']['params']['seq_length']}, ")
print(f"  feature_size={configs['model']['params']['feature_size']}, ")
print(f"  timesteps={configs['model']['params']['timesteps']}, ")
print(f"  loss_type={configs['model']['params']['loss_type']}")
print(f"  max_epochs={configs['solver']['max_epochs']}")

In [ ]:
# ---------- 5.2 实例化模型 ----------

model = instantiate_from_config(configs['model']).to(device)
model.fast_sampling = True
print(f"Model params: {get_model_parameters_info(model)}")
print(f"Model on device: {next(model.parameters()).device}")

In [ ]:
# ---------- 5.3 构建训练 dataloader ----------

dataloader_info = build_dataloader(configs, args)
dataloader = dataloader_info['dataloader']
train_dataset = dataloader_info['dataset']
print(f"Train dataset: {len(train_dataset)} samples, window={train_dataset.window}, features={train_dataset.var_num}")

In [ ]:
# ---------- 5.4 训练 ----------
# 注意：2000 epoch 在 GPU 上需要一定时间，请确保有足够的耐心。
# 如果希望快速测试，可减少 max_epochs（但会影响结果质量）。

# 创建 Trainer
trainer = Trainer(config=configs, args=args, model=model,
                  dataloader={'dataloader': dataloader})

# 开始训练
trainer.train()

In [ ]:
# ---------- 5.5 构建测试 dataloader ----------

args.mode = 'predict'
args.pred_len = seq_len
test_dataloader_info = build_dataloader_cond(configs, args)
test_scaled = test_dataloader_info['dataset'].samples
scaler = test_dataloader_info['dataset'].scaler
seq_length, feat_num = seq_len * 2, test_scaled.shape[-1]
pred_length = seq_len
real = test_scaled
test_dataset = test_dataloader_info['dataset']
test_dataloader = test_dataloader_info['dataloader']
print(f"Test dataset: {len(test_dataset)} samples")

In [ ]:
# ---------- 5.6 采样预测 & 评估 ----------

# Paper: metrics averaged over 10 sampling runs
mse_runs, mae_runs = [], []

for run in range(10):
    torch.manual_seed(2023 + run)
    np.random.seed(2023 + run)
    random.seed(2023 + run)
    sample, real_ = trainer.sample_forecast(test_dataloader, shape=[seq_len, feat_num])
    mse_runs.append(mean_squared_error(sample.reshape(-1), real_.reshape(-1)))
    mae_runs.append(mean_absolute_error(sample.reshape(-1), real_.reshape(-1)))

mse, mae = float(np.mean(mse_runs)), float(np.mean(mae_runs))
print(f"\n{'='*50}")
print(f"ARMD Stock 预测结果（10 次平均）:")
print(f"  MSE = {mse:.4f}")
print(f"  MAE = {mae:.4f}")
print(f"{'='*50}")
print("\nPaper reference (Table 1, ARMD on Stock, z-score):")
print("  MSE=0.235, MAE=0.269 (https://arxiv.org/abs/2412.09328)")

---
## 结束